# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abcreativeakshay/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

---

**My answer — Lane 2: Refresh / Content Opportunity Scoring.**

I am picking Lane 2: ranking existing content pages for refresh review — which pages should an editor look at first for a refresh, an expansion, or continued monitoring. I picked it for three reasons. First, the decision behind it is concrete and weekly: a content team really does sit down and choose which pages to work on next, so my output has a clear user from day one. Second, it runs on the starter dataset (30,000 pseudonymized pages, 32 clients), which I can inspect end to end and validate honestly before touching the bigger warehouse release. Third, it is a ranking problem with an observed outcome and a clean evaluation metric (Precision@K on a client holdout), and the repo ships a transparent rule baseline I can compare against — so I can measure whether a learned model actually earns its keep instead of just claiming it does. The code cell below loads the data and confirms the shape of the problem I am signing up for.


In [1]:
import os
import numpy as np
import pandas as pd

# Find the repo root from wherever this kernel started (works locally and in Colab).
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Working dir:", os.getcwd())
print("Rows x columns:", df.shape)
print("Pseudonymized clients:", df["client_id"].nunique())
print("Content types:", df["content_type"].nunique())
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # the label, derived in the open
print("Declining pages (trend_direction == 'down'):", int(df["is_declining_label"].sum()))


Working dir: /home/hatch/workspace/flyrank-ml
Rows x columns: (30000, 44)
Pseudonymized clients: 32
Content types: 3
Declining pages (trend_direction == 'down'): 16262


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

---

**My answer.**

**The decision:** every week, which pages go to the top of the content team's refresh-review queue — and which can safely wait. My work improves that ordering.

**Who acts:** a content editor (or SEO lead) works the queue top-down. For each page they pick an action — refresh it, expand it, review its title/CTR, or leave it on monitoring. Nobody re-sorts 30,000 pages by hand; the queue *is* the decision.

**The cost of a wrong call:** a false positive (a page flagged urgent that is not actually declining) costs editor hours — the scarcest resource in this system — and a needless refresh can disturb a page that was performing fine. A false negative (a declining page with real traffic that the queue buries) costs search visibility that compounds week after week while nobody looks at it. The costs are asymmetric in a specific way: because the editor only ever reaches the top of the queue, what matters most is that the *top* of the ranking is right. The code cell below puts numbers on why an unranked list is not an option.


In [2]:
# The capacity problem: how much declining content is there vs how fast can humans review it?
declining = df[df["is_declining_label"] == 1]
visible_declining = declining[declining["impressions_90d"] >= 100]

print(f"Pages total:                 {len(df):,}")
print(f"Declining pages:             {len(declining):,}  ({df['is_declining_label'].mean():.1%} of all pages)")
print(f"Declining AND visible (>=100 impressions in 90d): {len(visible_declining):,}")

for pace in (50, 200):
    weeks = len(declining) / pace
    print(f"Reviewing every declining page at {pace}/week would take ~{weeks:,.0f} weeks")

print()
print("So 'review everything' is not a plan. A ranked queue of the top ~50 per week is the")
print("realistic unit of work - which makes the quality of the top of the ranking the whole game.")


Pages total:                 30,000
Declining pages:             16,262  (54.2% of all pages)
Declining AND visible (>=100 impressions in 90d): 13,152
Reviewing every declining page at 50/week would take ~325 weeks
Reviewing every declining page at 200/week would take ~81 weeks

So 'review everything' is not a plan. A ranked queue of the top ~50 per week is the
realistic unit of work - which makes the quality of the top of the ranking the whole game.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

---

**My answer — three measured numbers from the starter data (code below):**

1. **The problem is big.** 16,262 of 30,000 pages (54.2%) are measured as declining (`trend_direction == "down"`). This is not a niche cleanup task; over half the catalogue is sliding.
2. **The decline is happening where the traffic is.** Declining pages still hold 51.3% of all impressions in the last 90 days, and their median impressions (961) are about double those of non-declining pages (472). These are pages people actually find — letting them slide has a real audience cost.
3. **The obvious rule misses almost everything.** Only 174 pages are "stale" in the classic sense (not updated in 180+ days), while 51.1% of pages updated within the last 30 days are *also* declining. A "just review the stale pages" rule would look at a nearly empty list while most of the decline happens elsewhere. Prioritisation has to come from measured signals, not one threshold.


In [3]:
# The three numbers behind section 3.
declining = df[df["is_declining_label"] == 1]
rest = df[df["is_declining_label"] == 0]

print(f"1) Declining share: {len(declining):,} / {len(df):,} pages = {df['is_declining_label'].mean():.1%}")
print(f"2) Share of all 90d impressions sitting on declining pages: "
      f"{declining['impressions_90d'].sum() / df['impressions_90d'].sum():.1%}")
print(f"   Median impressions_90d - declining: {declining['impressions_90d'].median():,.0f} "
      f"vs non-declining: {rest['impressions_90d'].median():,.0f}")

fresh = df[df["freshness_tier"] == "0-30"]
stale = df[df["days_since_last_update"] >= 180]
print(f"3) Pages updated in the last 30 days: {len(fresh):,} - of those, "
      f"{(fresh['is_declining_label'].mean()):.1%} are still declining")
print(f"   Pages not updated for 180+ days (the classic 'stale' rule): only {len(stale):,}")


1) Declining share: 16,262 / 30,000 pages = 54.2%
2) Share of all 90d impressions sitting on declining pages: 51.3%
   Median impressions_90d - declining: 961 vs non-declining: 472


3) Pages updated in the last 30 days: 20,480 - of those, 51.1% are still declining
   Pages not updated for 180+ days (the classic 'stale' rule): only 174


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

---

**My answer.**

**What my work will be able to say** (and the words I will use): *observed* and *measured* patterns in this anonymized trailing-90-day snapshot; *directional* signals about which pages look most at risk; and a *decision-support* ranking that helps an editor spend limited review time where the measured evidence is strongest. If a model beats the rule baseline on clients it has never seen, I can say exactly that, with the number attached.

**What it never will say:** that refreshing a page *causes* it to recover (I have no experiment or before/after design for that); anything about "predicting Google's algorithm"; anything about a real client — the IDs in this data are pseudonyms for grouping and splitting only, never identities. One more honesty point I will keep visible in every later notebook: my label describes decline *measured in the observed window*, not a guaranteed future decline.

**The discipline that keeps the claims honest:** the label is derived from `trend_direction` / `trend_pct`, so those two columns can never be model features — that would be grading my own homework. The check below shows how seductive that leak is: the sign of `trend_pct` alone "predicts" the label at 88.5% accuracy, because every declining page has `trend_pct < 0` by construction. A model fed that column would look brilliant and mean nothing.


In [4]:
# The leakage check behind section 4: why trend_pct / trend_direction are NEVER features.
leaky_rule = (df["trend_pct"] < 0).astype(int)
acc = (leaky_rule == df["is_declining_label"]).mean()
print(f"Accuracy of the one-line rule 'trend_pct < 0 -> declining': {acc:.1%}")
print()
print(pd.crosstab(df["trend_direction"], df["trend_pct"] < 0, margins=True))
print()
print("Every 'down' row has trend_pct < 0 and no 'down' row has trend_pct >= 0:")
print("the label is computed FROM trend_pct, so feeding it to a model is leakage, not skill.")


Accuracy of the one-line rule 'trend_pct < 0 -> declining': 88.5%

trend_pct        False   True    All
trend_direction                     
down                 0  16262  16262
flat              1152      0   1152
new               2236      0   2236
stable            2509   3453   5962
up                4388      0   4388
All              10285  19715  30000

Every 'down' row has trend_pct < 0 and no 'down' row has trend_pct >= 0:
the label is computed FROM trend_pct, so feeding it to a model is leakage, not skill.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.